# 👟 YOLOv8n-Pose (4 Coarse Keypoints) Two-Stage Training Pipeline
### Stage 1: Pretraining on `shoes_v2` → Stage 2: Fine-Tuning on `shuffled_v3`

This notebook trains the **YOLOv8n-Pose (4 Coarse Keypoints)** detector on Google Colab (A100 / T4 GPU) with full **Google Drive synchronization**, two-stage transfer learning, ONNX export (FP32, FP16, INT8), and video evaluation.

---
### 📌 Workflow Overview:
1. **Step 1:** Mount Google Drive for persistent storage of datasets, checkpoints, and models.
2. **Step 2:** Clone GitHub repository (`yolo_stage1` branch) & install Ultralytics + dependencies.
3. **Step 3:** Convert datasets into 4-KP native format (`shoes_v2_4kp` and `shuffled_v3_4kp`).
4. **Step 4:** **Stage 1 Pretraining:** Train `yolov8n-pose.pt` on `shoes_v2_4kp` (100 epochs).
5. **Step 5:** **Stage 2 Fine-Tuning:** Fine-tune Stage 1 best weights on `shuffled_v3_4kp` (150 epochs).
6. **Step 6:** **Model Evaluation:** Evaluate final model on test split (mAP50, Pose mAP).
7. **Step 7:** **ONNX Export:** Export to FP32, FP16 (WebGPU/CoreML), and INT8 (WASM) with shape verification `[1, 18, 2100]`.
8. **Step 8:** **Video Inference:** Run inference on `test_video.mp4` with real-time 4-KP HUD overlay.
9. **Step 9:** Download trained models to local machine (optional).

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create experiment directories in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/Shoes_VTO_Experiments'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f"✅ Google Drive Mounted! Permanent storage directory: {DRIVE_DIR}")

### Step 2: Clone GitHub Repository & Install Dependencies

In [ ]:
%cd /content
!rm -rf Shoes_VTO
!git clone -b yolo_stage1 https://github.com/HagAli22/Shoes_VTO.git
%cd Shoes_VTO

# Install Ultralytics and export dependencies
!pip install -q ultralytics albumentations onnx onnxruntime onnxsim onnxconverter-common opencv-python-headless tqdm

import torch, ultralytics
print(f"\nPyTorch Version     : {torch.__version__}")
print(f"Ultralytics Version : {ultralytics.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Step 3: Prepare & Convert Datasets to 4-KP Format
*(Extracts 4 coarse keypoints: Toe, Heel, Ball Medial, Ball Lateral & creates standardized configs)*

In [ ]:
import os

# Run dataset converter for both shoes_v2 and shuffled_v3
!python tools/convert_dataset_to_4kp.py --all

# Display summary of converted datasets
print("\n--- Converted 4-KP Dataset Summary ---")
for ds in ["shoes_v2_4kp", "shuffled_v3_4kp"]:
    print(f"\nDataset: {ds}")
    for split in ["train", "valid", "test"]:
        img_dir = f"data/{ds}/{split}/images"
        if os.path.exists(img_dir):
            print(f"  {split:5s} images: {len(os.listdir(img_dir))}")

### Step 4: Stage 1 Pretraining on `shoes_v2_4kp`
*(Initializes pretrained `yolov8n-pose.pt` on the coarse dataset for initial weight convergence)*

In [ ]:
!yolo pose train \
  data=configs/shoes_v2_4kp.yaml \
  model=yolov8n-pose.pt \
  epochs=100 \
  imgsz=320 \
  batch=32 \
  device=0 \
  optimizer=AdamW \
  lr0=0.002 \
  patience=25 \
  project=outputs/stage1_4kp \
  name=pretrain_shoes_v2

# Backup Stage 1 best checkpoint to Google Drive
!cp outputs/stage1_4kp/pretrain_shoes_v2/weights/best.pt "{DRIVE_DIR}/checkpoints/yolo_4kp_stage1_pretrain.pt"
print(f"\n✅ Stage 1 checkpoint saved to: {DRIVE_DIR}/checkpoints/yolo_4kp_stage1_pretrain.pt")

### Step 5: Stage 2 Fine-Tuning on `shuffled_v3_4kp`
*(Fine-tunes Stage 1 weights on the clean, uniformly balanced 1,102-image dataset)*

In [ ]:
!yolo pose train \
  data=configs/shuffled_v3_4kp.yaml \
  model=outputs/stage1_4kp/pretrain_shoes_v2/weights/best.pt \
  epochs=150 \
  imgsz=320 \
  batch=32 \
  device=0 \
  optimizer=AdamW \
  lr0=0.001 \
  patience=35 \
  project=outputs/stage1_4kp \
  name=finetune_shuffled_v3

# Backup final fine-tuned checkpoint to Google Drive
!cp outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt "{DRIVE_DIR}/checkpoints/yolo_4kp_best.pt"
print(f"\n✅ Stage 2 fine-tuned checkpoint saved to: {DRIVE_DIR}/checkpoints/yolo_4kp_best.pt")

### Step 6: Validate Final Model on Test Split

In [ ]:
!yolo pose val \
  data=configs/shuffled_v3_4kp.yaml \
  model=outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt \
  imgsz=320 \
  split=test \
  device=0

### Step 7: Export to ONNX (FP32, FP16, INT8) & Synchronize with Drive

In [ ]:
!python -m src.export.export_yolo_4kp \
  --weights outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt \
  --output_dir deliverables/stage_a_4kp \
  --prefix stage-a-320-yolo4kp

# Copy exported models to Google Drive
!cp -r deliverables/stage_a_4kp/* "{DRIVE_DIR}/models/"
print(f"\n✅ Exported ONNX models synchronized to Google Drive: {DRIVE_DIR}/models/")
!ls -lh "{DRIVE_DIR}/models/"

### Step 8: Test Video Inference & Visual Benchmark (`test_video.mp4`)
*(Generates video with bounding boxes, 4-KP skeleton overlay, and real-time latency HUD)*

In [ ]:
import os

# Search candidate video locations
video_candidates = [
    f'{DRIVE_DIR}/test_video.mp4',
    'deliverables/stage_a_16kp/test_video.mp4',
    'data/test_video.mp4'
]

video_path = next((vc for vc in video_candidates if os.path.exists(vc)), None)

if video_path is None:
    print("❌ test_video.mp4 not found. Please upload test_video.mp4 to Google Drive at:", f'{DRIVE_DIR}/test_video.mp4')
else:
    print(f"🎥 Running video evaluation on: {video_path}")
    out_video_drive = f'{DRIVE_DIR}/yolo_4kp_annotated_video.mp4'
    
    !python tools/evaluation/eval_yolo_4kp_video.py \
      --video "{video_path}" \
      --model outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt \
      --output "{out_video_drive}" \
      --conf 0.25 \
      --device 0
    
    print(f"\n✅ Annotated output video saved directly to Google Drive: {out_video_drive}")

### Step 9: Download Artifacts to Local PC (Optional)

In [ ]:
from google.colab import files

# Download the FP32 and FP16 models + best checkpoint
try:
    files.download('deliverables/stage_a_4kp/stage-a-320-yolo4kp-fp32.onnx')
    files.download('deliverables/stage_a_4kp/stage-a-320-yolo4kp-fp16.onnx')
    files.download('outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt')
except Exception as e:
    print(f"Note: {e}")